In [1]:
import numpy as np

def max_pool2d(input, kernel_size, stride=1, padding=0):
    """
    手动实现二维最大池化
    
    参数:
        input: (H, W) 或 (C, H, W) 的 numpy 数组
        kernel_size: 池化窗口大小 (int 或 tuple)
        stride: 步幅 (int 或 tuple)
        padding: 填充 (int 或 tuple)
    
    返回:
        output: 池化后的结果
    """
    # 统一处理为 tuple
    if isinstance(kernel_size, int):
        kernel_size = (kernel_size, kernel_size)
    if isinstance(stride, int):
        stride = (stride, stride)
    if isinstance(padding, int):
        padding = (padding, padding)
    
    # 处理输入维度
    if input.ndim == 2:
        input = input[np.newaxis, :, :]
        add_dim = True
    else:
        add_dim = False
    
    C, H, W = input.shape
    KH, KW = kernel_size
    SH, SW = stride
    PH, PW = padding
    
    # 添加填充
    input_pad = np.pad(input, 
                       ((0, 0), (PH, PH), (PW, PW)),
                       mode='constant', constant_values=0)
    
    # 计算输出尺寸
    H_out = (H + 2*PH - KH) // SH + 1
    W_out = (W + 2*PW - KW) // SW + 1
    
    # 初始化输出
    output = np.zeros((C, H_out, W_out))
    
    # 执行池化
    for c in range(C):
        for i in range(H_out):
            for j in range(W_out):
                h_start = i * SH
                h_end = h_start + KH
                w_start = j * SW
                w_end = w_start + KW
                window = input_pad[c, h_start:h_end, w_start:w_end]
                output[c, i, j] = np.max(window)
    
    if add_dim:
        output = output[0]
    
    return output

# 测试代码
if __name__ == "__main__":
    # 创建测试数据
    x = np.array([
        [1, 2, 3, 4],
        [5, 6, 7, 8],
        [9, 10, 11, 12],
        [13, 14, 15, 16]
    ])
    
    print("输入:")
    print(x)
    print()
    
    # 2x2 池化，stride=2，padding=0
    out = max_pool2d(x, kernel_size=2, stride=2)
    print("2x2 Max Pooling (stride=2):")
    print(out)
    print()
    
    # 2x2 池化，stride=1，padding=0
    out2 = max_pool2d(x, kernel_size=2, stride=1)
    print("2x2 Max Pooling (stride=1):")
    print(out2)

输入:
[[ 1  2  3  4]
 [ 5  6  7  8]
 [ 9 10 11 12]
 [13 14 15 16]]

2x2 Max Pooling (stride=2):
[[ 6.  8.]
 [14. 16.]]

2x2 Max Pooling (stride=1):
[[ 6.  7.  8.]
 [10. 11. 12.]
 [14. 15. 16.]]


In [2]:
import torch
import torch.nn as nn

class NiNBlock(nn.Module):
    """
    NiN 块：一个普通卷积层 + 两个 1x1 卷积层
    """
    def __init__(self, in_channels, out_channels, kernel_size, stride, padding):
        super(NiNBlock, self).__init__()
        self.block = nn.Sequential(
            # 普通卷积层
            nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding),
            nn.ReLU(),
            # 第一个 1x1 卷积
            nn.Conv2d(out_channels, out_channels, kernel_size=1),
            nn.ReLU(),
            # 第二个 1x1 卷积
            nn.Conv2d(out_channels, out_channels, kernel_size=1),
            nn.ReLU()
        )
    
    def forward(self, x):
        return self.block(x)

# 测试
if __name__ == "__main__":
    # 创建 NiN 块：输入3通道，输出16通道，3x3卷积，stride=1，padding=1
    nin_block = NiNBlock(in_channels=3, out_channels=16, 
                         kernel_size=3, stride=1, padding=1)
    
    # 测试输入
    x = torch.randn(1, 3, 32, 32)
    out = nin_block(x)
    print(f"输入形状: {x.shape}")
    print(f"输出形状: {out.shape}")

输入形状: torch.Size([1, 3, 32, 32])
输出形状: torch.Size([1, 16, 32, 32])


In [3]:
import torch
import torch.nn as nn

class Residual(nn.Module):
    """
    残差块：两个 3x3 卷积层 + 批量归一化层
    """
    def __init__(self, in_channels, out_channels, use_1x1conv=False, stride=1):
        super(Residual, self).__init__()
        
        # 第一个卷积层
        self.conv1 = nn.Conv2d(in_channels, out_channels, 
                               kernel_size=3, stride=stride, padding=1)
        self.bn1 = nn.BatchNorm2d(out_channels)
        
        # 第二个卷积层
        self.conv2 = nn.Conv2d(out_channels, out_channels,
                               kernel_size=3, stride=1, padding=1)
        self.bn2 = nn.BatchNorm2d(out_channels)
        
        # 如果需要调整输入形状（1x1卷积）
        if use_1x1conv:
            self.shortcut = nn.Conv2d(in_channels, out_channels,
                                      kernel_size=1, stride=stride)
        else:
            self.shortcut = None
        
        self.relu = nn.ReLU()
    
    def forward(self, x):
        # 残差路径
        y = self.relu(self.bn1(self.conv1(x)))
        y = self.bn2(self.conv2(y))
        
        # 恒等映射路径
        if self.shortcut is not None:
            x = self.shortcut(x)
        
        # 相加 + ReLU
        out = self.relu(y + x)
        return out

# 测试
if __name__ == "__main__":
    # 情况1：输入输出通道相同，不需要 1x1 卷积
    block1 = Residual(16, 16, use_1x1conv=False)
    x = torch.randn(1, 16, 32, 32)
    out1 = block1(x)
    print(f"输入相同通道: {x.shape} → {out1.shape}")
    
    # 情况2：输入输出通道不同，需要 1x1 卷积
    block2 = Residual(16, 32, use_1x1conv=True, stride=2)
    x2 = torch.randn(1, 16, 32, 32)
    out2 = block2(x2)
    print(f"输入不同通道: {x2.shape} → {out2.shape}")

输入相同通道: torch.Size([1, 16, 32, 32]) → torch.Size([1, 16, 32, 32])
输入不同通道: torch.Size([1, 16, 32, 32]) → torch.Size([1, 32, 16, 16])


In [4]:
import torchvision.transforms as transforms
from PIL import Image
import torch

# 创建图像增广管道
augmentation_pipeline = transforms.Compose([
    # 1. 随机裁剪并缩放到 224x224
    transforms.RandomResizedCrop(224, scale=(0.08, 1.0)),
    
    # 2. 50% 概率水平翻转
    transforms.RandomHorizontalFlip(p=0.5),
    
    # 3. 随机改变亮度、对比度、饱和度
    transforms.ColorJitter(brightness=0.5, contrast=0.5, saturation=0.5),
    
    # 4. 转换为 Tensor
    transforms.ToTensor()
])

# 测试
if __name__ == "__main__":
    # 加载一张测试图片（需要你自己放一张图片）
    # img = Image.open("test.jpg")
    
    # 模拟一张图片
    img = Image.new('RGB', (500, 500), color='red')
    
    # 应用增广
    augmented = augmentation_pipeline(img)
    print(f"增广后形状: {augmented.shape}")
    print("管道定义完成 ✓")

增广后形状: torch.Size([3, 224, 224])
管道定义完成 ✓


In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F

def label_smoothing_cross_entropy(logits, labels, epsilon=0.1):
    """
    标签平滑后的交叉熵损失
    
    参数:
        logits: 模型输出 (batch_size, num_classes)
        labels: 真实标签 (batch_size,)
        epsilon: 平滑因子
    """
    num_classes = logits.shape[-1]
    batch_size = labels.shape[0]
    
    # 使用 log_softmax 获得对数概率
    log_probs = F.log_softmax(logits, dim=-1)
    
    # 构造平滑标签
    # 真实类别: 1 - epsilon
    # 其他类别: epsilon / (num_classes - 1)
    smooth_labels = torch.full_like(log_probs, epsilon / (num_classes - 1))
    smooth_labels.scatter_(1, labels.unsqueeze(1), 1 - epsilon)
    
    # 计算损失： -sum(平滑标签 × log_probs) / batch_size
    loss = -torch.sum(smooth_labels * log_probs) / batch_size
    return loss

# 测试
if __name__ == "__main__":
    # 假设 5 分类，batch_size=3
    logits = torch.randn(3, 5)
    labels = torch.tensor([0, 2, 4])
    
    # 普通交叉熵
    ce_loss = F.cross_entropy(logits, labels)
    
    # 标签平滑交叉熵
    smooth_loss = label_smoothing_cross_entropy(logits, labels, epsilon=0.1)
    
    print(f"普通交叉熵损失: {ce_loss.item():.4f}")
    print(f"标签平滑交叉熵: {smooth_loss.item():.4f}")

普通交叉熵损失: 2.5999
标签平滑交叉熵: 2.5137
